# 5c. Check sensitivity to label attribution

This notebook examines how the interpretation of scenario 3 affects the validation labels and detector-selection results.

The as-run attribution included accounts involved in the mass-email reaction chain. The supporting analysis distinguishes designated insiders and the masqueraded supervisor from accounts treated as bystanders under the documented correction.

**Inputs:** the saved scores and comparison table from Notebook 5, the policy and modelling module from Notebook 4, and the insider and organisational records from Notebook 1.

**Outputs:** `label_attribution_anatomy.csv`, `sensitivity_v1_selection.csv`, `sensitivity_detector_robustness.csv` and `sensitivity_summary.json`.

No detector is refitted or rescored. The notebook compares existing scores under different label views and reports detector-family sensitivity. It retains the original primary comparison rather than replacing the selected detector after test inspection.

The account categories implement the documented correction. This notebook does not independently re-audit every underlying scenario event.

*Documentation copy, 8 November 2026. Notes edited for clarity; code, saved outputs and execution metadata are unchanged. This copy has not been rerun.*


In [1]:
# ============================================================
# 0. Setup - committed artefacts only; NOTHING is fitted here
# ============================================================
import os, glob, json, hashlib, warnings
import numpy as np
import pandas as pd

SEED = 42
warnings.filterwarnings("ignore")
WORK = "/kaggle/working"

def find(pattern, required=True, what=""):
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    assert hits or not required, f"Required input not attached: {pattern} ({what})"
    return hits[0] if hits else None

P_SCORES = find("scores_all_models.parquet",     what="NB5 committed output")
P_CMP    = find("comparison_test_at_budget.csv", what="NB5 committed output")
P_UTILS  = find("modelling_utils.py",            what="NB4 output")
P_POLICY = find("threshold_policy.json",         what="NB4 output")
P_INS    = find("insiders.parquet",              what="NB1 output")
P_LDAP   = find("ldap_monthly.parquet",          what="NB1 output")

import importlib.util
spec = importlib.util.spec_from_file_location("modelling_utils", P_UTILS)
mu = importlib.util.module_from_spec(spec); spec.loader.exec_module(mu)
UTILS_SHA = hashlib.sha256(open(P_UTILS, "rb").read()).hexdigest()[:16]

POLICY = json.load(open(P_POLICY))
DET, K = POLICY["detector"], float(POLICY["k_pct"])
print(f"utils sha256[:16]={UTILS_SHA} | FROZEN policy: detector={DET}, k={K}%")

scores = pd.read_parquet(P_SCORES)
scores["user"] = scores["user"].astype(str)
scores["day"]  = pd.to_datetime(scores["day"])
CONFIGS   = ["M0", "M1", "M2", "M3"]
DETECTORS = ["if", "ae", "ens"]
assert all(f"{c}_{d}" in scores.columns for c in CONFIGS for d in DETECTORS)
print(f"committed scores: {len(scores):,} user-days | "
      f"positives per partition: {scores.groupby('partition')['label'].sum().to_dict()}")

utils sha256[:16]=147d19b0977c79e8 | FROZEN policy: detector=if, k=1.0%
committed scores: 1,393,997 user-days | positives per partition: {'test': 38, 'train': 0, 'val': 40}


## 1. Check the reconstructed operating point

Rebuild the fixed-budget alerts from the saved scores and compare their true-positive and false-positive counts with the retained Notebook 5 table.

Matching totals show agreement at the operating point. The same score records, row ordering and ranking rule are also needed to reproduce alert membership; equality of counts alone is not sufficient.


In [2]:
# ============================================================
# 1. Provenance gate - reconstructed alert sets == committed
# ============================================================
test = scores[scores.partition == "test"].reset_index(drop=True)
yt_asrun = test["label"].to_numpy()
ut       = test["user"].to_numpy()

ALERT_FROZEN = {c: mu.alerts_at_budget(test[f"{c}_{DET}"].to_numpy(), K) for c in CONFIGS}
chk = pd.read_csv(P_CMP)
for c in CONFIGS:
    tp, fp, fn, tn = mu.counts(yt_asrun, ALERT_FROZEN[c])
    row = chk[chk.config == c].iloc[0]
    assert (tp == int(row.tp)) and (fp == int(row.fp)), (
        f"{c}: reconstructed {tp}/{fp} != committed {int(row.tp)}/{int(row.fp)}")
    print(f"  {c}: TP={tp}  FP={fp:,}  == committed OK")
print("\nPROVENANCE GATE PASSED - alert sets identical to the committed NB5 run.")

  M0: TP=5  FP=3,950  == committed OK
  M1: TP=4  FP=3,951  == committed OK
  M2: TP=5  FP=3,950  == committed OK
  M3: TP=5  FP=3,950  == committed OK

PROVENANCE GATE PASSED - alert sets identical to the committed NB5 run.


## 2. Separate the label-attribution groups

The analysis uses three categories:

1. **Designated insider:** an account listed in `insiders.parquet`.
2. **Masqueraded supervisor:** the supervisor account identified through the scenario-3 insider's LDAP record and retained as compromised under the documented interpretation.
3. **Bystander:** another account with an as-run positive label, treated as part of the ordinary mass-email reaction under that correction.

The supervisor is resolved from the insider record, supervisor name, employee name and account identifier rather than a hard-coded account ID. The categories depend on the scenario interpretation; identifying an account relationship does not independently verify the meaning of each event.

The table records how these categories contribute to the as-run positive days.


In [3]:
# ============================================================
# 2. Derive the corrected label set (data-driven, no hard-coded IDs)
# ============================================================
ins  = pd.read_parquet(P_INS);  ins["user"] = ins["user"].astype(str)
ldap = pd.read_parquet(P_LDAP); ldap["user_id"] = ldap["user_id"].astype(str)

DESIGNATED = set(ins["user"])
s3 = ins.loc[ins["scenario"].astype(int) == 3, "user"]
assert len(s3) == 1, f"expected exactly one scenario-3 insider, found {list(s3)}"
S3_USER  = s3.iloc[0]
SUP_NAME = ldap.loc[ldap.user_id == S3_USER, "supervisor"].dropna().iloc[0]
MASQ     = set(ldap.loc[ldap.employee_name == SUP_NAME, "user_id"])
assert MASQ, f"could not resolve supervisor '{SUP_NAME}' to a user_id"
print(f"scenario-3 insider : {S3_USER}")
print(f"LDAP supervisor    : {SUP_NAME} -> account(s) {sorted(MASQ)}")
print(f"designated insiders: {sorted(DESIGNATED)}")

CORRECTED_ACCOUNTS = DESIGNATED | MASQ

def classify(u):
    if u in DESIGNATED: return "designated_insider"
    if u in MASQ:       return "masqueraded_supervisor"
    return "bystander"

pos = scores[scores.label == 1].copy()
pos["attribution"] = pos["user"].map(classify)
anatomy = (pd.crosstab(pos["partition"], pos["attribution"])
             .reindex(columns=["designated_insider", "masqueraded_supervisor",
                               "bystander"], fill_value=0))
anatomy["total"] = anatomy.sum(axis=1)
anatomy.to_csv(f"{WORK}/label_attribution_anatomy.csv")
print("\nmalicious user-days by attribution class:")
print(anatomy.to_string())

n_by = int(pos[pos.attribution == "bystander"]["user"].nunique())
print(f"\nbystander accounts: {n_by} | "
      f"share of validation positives that are bystander days: "
      f"{anatomy.loc['val','bystander'] / anatomy.loc['val','total']:.1%}")

# corrected label vector, aligned to `scores`
scores["label_corrected"] = ((scores.label == 1) &
                             (scores.user.isin(CORRECTED_ACCOUNTS))).astype("int8")
assert int(scores.loc[scores.partition == "train", "label_corrected"].sum()) == 0, \
    "benign-train premise violated under corrected labels"
print("\ncorrected positives per partition:",
      scores.groupby("partition")["label_corrected"].sum().to_dict())

scenario-3 insider : PLJ1771
LDAP supervisor    : Hedda Indira Savage -> account(s) ['HIS1706']
designated insiders: ['ACM2278', 'CDE1846', 'CMP2946', 'MBG3183', 'PLJ1771']

malicious user-days by attribution class:
attribution  designated_insider  masqueraded_supervisor  bystander  total
partition                                                                
test                         38                       0          0     38
val                           6                       3         31     40

bystander accounts: 23 | share of validation positives that are bystander days: 77.5%

corrected positives per partition: {'test': 38, 'train': 0, 'val': 9}


## 3. Recalculate the validation selection statistic

Apply Notebook 4's maximum-average-precision selection rule to the same saved scores under three label views: as-run, insiders plus the masqueraded supervisor, and designated insiders only.

The legacy `pr_auc` field contains average precision. The designated-insider-only view is a narrower sensitivity comparison; it is not a replacement for the study's scope, which includes compromised legitimate accounts.

The comparison shows whether the recorded detector ordering depends on outcome attribution. It does not refit or reselect the primary detector for the completed experiment.


In [4]:
# ============================================================
# 3. V1 recomputation - validation PR-AUC by detector family
# ============================================================
from sklearn.metrics import roc_auc_score, average_precision_score

val = scores[scores.partition == "val"].reset_index(drop=True)
LABEL_SETS = {
    "as_run":              val["label"].to_numpy(),
    "insiders_supervisor": val["label_corrected"].to_numpy(),
    "insiders_only":       ((val.label == 1) & val.user.isin(DESIGNATED)).to_numpy().astype(int),
}

rows = []
for name, y in LABEL_SETS.items():
    for det in DETECTORS:
        s = val[f"M0_{det}"].to_numpy()
        rows.append({"label_set": name, "n_positives": int(y.sum()), "detector": det,
                     "val_pr_auc": average_precision_score(y, s),
                     "val_roc_auc": roc_auc_score(y, s)})
v1 = pd.DataFrame(rows)
v1["v1_selected"] = v1.groupby("label_set")["val_pr_auc"].transform("max") == v1["val_pr_auc"]
v1.to_csv(f"{WORK}/sensitivity_v1_selection.csv", index=False)
print("V1 detector selection (argmax validation PR-AUC):")
print(v1.to_string(index=False))
for name in LABEL_SETS:
    sel = v1.loc[v1.label_set == name].sort_values("val_pr_auc", ascending=False).iloc[0]
    print(f"  {name:22s} ({sel.n_positives:2d} pos) -> V1 selects {sel.detector.upper()}")

V1 detector selection (argmax validation PR-AUC):
          label_set  n_positives detector  val_pr_auc  val_roc_auc  v1_selected
             as_run           40       if    0.000836     0.829724         True
             as_run           40       ae    0.000808     0.778565        False
             as_run           40      ens    0.000704     0.808532        False
insiders_supervisor            9       if    0.000242     0.860358        False
insiders_supervisor            9       ae    0.001593     0.855728         True
insiders_supervisor            9      ens    0.000337     0.864392        False
      insiders_only            6       if    0.000323     0.950120        False
      insiders_only            6       ae    0.002351     0.942136         True
      insiders_only            6      ens    0.000464     0.951867        False
  as_run                 (40 pos) -> V1 selects IF
  insiders_supervisor    ( 9 pos) -> V1 selects AE
  insiders_only          ( 6 pos) -> V1 selects 

## 4. Compare the retained detector families

Evaluate the M0–M2 fixed-budget comparison under Isolation Forest, the reconstruction autoencoder and their ensemble, using the saved scores and paired user-cluster resampling.

The validation correction leaves the test labels unchanged, but it changes the validation selection statistic. Agreement in the eventual hypothesis outcome would not make that selection defect merely a reporting issue or erase the earlier test inspection.

Read the sensitivity results alongside the primary comparison. The later matched-recall reconstruction for the alternative families is retained in Notebook 5e.


In [5]:
# ============================================================
# 4. Test verdict under every detector family (labels unchanged on test)
# ============================================================
test = scores[scores.partition == "test"].reset_index(drop=True)
assert (test["label"].to_numpy() == test["label_corrected"].to_numpy()).all(), \
    "test labels differ under correction - the robustness framing below does not hold"
yt = test["label"].to_numpy(); ut = test["user"].to_numpy()

def per_user_counts(users, y, alert):
    df = pd.DataFrame({"user": users, "y": np.asarray(y).astype(bool),
                       "a": np.asarray(alert).astype(bool)})
    return (df.assign(tp=df.y & df.a, fp=~df.y & df.a,
                      fn=df.y & ~df.a, tn=~df.y & ~df.a)
              .groupby("user")[["tp", "fp", "fn", "tn"]].sum())

def paired_recall_delta(users, y, alert_ref, alert_new, n_boot=1000, seed=SEED):
    A = per_user_counts(users, y, alert_ref)
    B = per_user_counts(users, y, alert_new).reindex(A.index).fillna(0)
    is_pos = (A["tp"] + A["fn"]) > 0
    Ap, An = A[is_pos].to_numpy("int64"), A[~is_pos].to_numpy("int64")
    Bp, Bn = B[is_pos].to_numpy("int64"), B[~is_pos].to_numpy("int64")
    ref = mu.metrics_from_counts(*[int(v) for v in A.sum()])
    new = mu.metrics_from_counts(*[int(v) for v in B.sum()])
    rng = np.random.default_rng(seed); d = []
    for _ in range(n_boot):
        ip = rng.integers(0, len(Ap), len(Ap)); ib = rng.integers(0, len(An), len(An))
        ca = Ap[ip].sum(axis=0) + An[ib].sum(axis=0)
        cb = Bp[ip].sum(axis=0) + Bn[ib].sum(axis=0)
        d.append(mu.metrics_from_counts(*[int(v) for v in cb])["recall"] -
                 mu.metrics_from_counts(*[int(v) for v in ca])["recall"])
    return (new["recall"] - ref["recall"],
            float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

rob = []
for det in DETECTORS:
    S = {c: test[f"{c}_{det}"].to_numpy() for c in CONFIGS}
    A = {c: mu.alerts_at_budget(S[c], K) for c in CONFIGS}
    m = {c: mu.metrics_from_counts(*mu.counts(yt, A[c])) for c in CONFIGS}
    d2, lo2, hi2 = paired_recall_delta(ut, yt, A["M0"], A["M2"])
    d3, lo3, hi3 = paired_recall_delta(ut, yt, A["M0"], A["M3"])
    rob.append({
        "detector": det, "frozen": det == DET,
        "M0_tp": m["M0"]["tp"], "M2_tp": m["M2"]["tp"], "M3_tp": m["M3"]["tp"],
        "M0_recall": m["M0"]["recall"], "M2_recall": m["M2"]["recall"],
        "M0_pr_auc": average_precision_score(yt, S["M0"]),
        "M2_pr_auc": average_precision_score(yt, S["M2"]),
        "critB_delta": d2, "critB_lo": lo2, "critB_hi": hi2,
        "critB_significant": bool(lo2 > 0),
        "m3_gate_delta": d3, "m3_gate_lo": lo3, "m3_gate_hi": hi3,
        "m3_leak_flag": bool(lo3 > 0),
    })
rob = pd.DataFrame(rob)
rob["verdict"] = np.where(rob.critB_significant, "H1 SUPPORTED", "H1 NOT SUPPORTED")
rob.to_csv(f"{WORK}/sensitivity_detector_robustness.csv", index=False)
print(f"TEST at k={K}%, every detector family (test labels unaffected by the correction):")
print(rob[["detector", "frozen", "M0_tp", "M2_tp", "M0_recall", "M2_recall",
           "critB_delta", "critB_lo", "critB_hi", "m3_leak_flag",
           "verdict"]].to_string(index=False))
print("\nverdict invariant across detector families:", rob.verdict.nunique() == 1)
print("negative control passes for every family:",  (~rob.m3_leak_flag).all())

TEST at k=1.0%, every detector family (test labels unaffected by the correction):
detector  frozen  M0_tp  M2_tp  M0_recall  M2_recall  critB_delta  critB_lo  critB_hi  m3_leak_flag          verdict
      if    True      5      5   0.131579   0.131579          0.0       0.0       0.0         False H1 NOT SUPPORTED
      ae   False      7      7   0.184211   0.184211          0.0       0.0       0.0         False H1 NOT SUPPORTED
     ens   False      6      6   0.157895   0.157895          0.0       0.0       0.0         False H1 NOT SUPPORTED

verdict invariant across detector families: True
negative control passes for every family: True


## 5. Save the sensitivity summary

Save the attribution, selection and robustness tables together with `sensitivity_summary.json`. The audit records the input checks and confirms that the detector scores were reused rather than recalculated.

These outputs document sensitivity within the retained experiment. They do not create a new independent test set or replace the Environment A hypothesis decision.


In [6]:
# ============================================================
# 5. Summary + audit
# ============================================================
summary = {
    "utils_sha256_16": UTILS_SHA,
    "frozen_policy": {"detector": DET, "k_pct": K},
    "scenario3_insider": S3_USER,
    "masqueraded_supervisor": sorted(MASQ),
    "n_bystander_accounts": n_by,
    "val_positives_asrun": int(anatomy.loc["val", "total"]),
    "val_positives_corrected": int(scores.loc[scores.partition == "val",
                                              "label_corrected"].sum()),
    "val_bystander_share": float(anatomy.loc["val", "bystander"] /
                                 anatomy.loc["val", "total"]),
    "test_labels_unchanged": True,
    "v1_selection": {n: str(v1.loc[v1.label_set == n]
                            .sort_values("val_pr_auc", ascending=False)
                            .iloc[0]["detector"]) for n in LABEL_SETS},
    "verdict_by_detector": dict(zip(rob.detector, rob.verdict)),
    "verdict_invariant": bool(rob.verdict.nunique() == 1),
    "negative_control_passes_all": bool((~rob.m3_leak_flag).all()),
    "seed": SEED,
}
json.dump(summary, open(f"{WORK}/sensitivity_summary.json", "w"), indent=2, default=str)

print("LABEL-ATTRIBUTION SENSITIVITY AUDIT")
print("-" * 66)
print("1. NOTHING fitted or scored: committed NB5 scores reused verbatim.")
print("2. Alert sets reconstructed by deterministic rank and ASSERTED equal to")
print(f"   the committed NB5 comparison table (all four configurations).")
print("3. Corrected labels derived from insiders.parquet + ldap_monthly.parquet;")
print("   no account identifier hard-coded.")
print("4. Benign-train premise re-asserted under the corrected label set.")
print("5. Test labels are IDENTICAL under both label sets - the canonical test")
print("   evaluation is unchanged; only V1 detector selection is affected.")
print(f"6. Verdict invariant across detector families: {summary['verdict_invariant']}")
print("-" * 66)
for f in sorted(os.listdir(WORK)):
    p = f"{WORK}/{f}"
    if os.path.isfile(p):
        print(f"  {f:42s} {os.path.getsize(p)/1e6:8,.3f} MB")

LABEL-ATTRIBUTION SENSITIVITY AUDIT
------------------------------------------------------------------
1. NOTHING fitted or scored: committed NB5 scores reused verbatim.
2. Alert sets reconstructed by deterministic rank and ASSERTED equal to
   the committed NB5 comparison table (all four configurations).
3. Corrected labels derived from insiders.parquet + ldap_monthly.parquet;
   no account identifier hard-coded.
4. Benign-train premise re-asserted under the corrected label set.
5. Test labels are IDENTICAL under both label sets - the canonical test
   evaluation is unchanged; only V1 detector selection is affected.
6. Verdict invariant across detector families: True
------------------------------------------------------------------
  __notebook__.ipynb                            0.029 MB
  label_attribution_anatomy.csv                 0.000 MB
  sensitivity_detector_robustness.csv           0.001 MB
  sensitivity_summary.json                      0.001 MB
  sensitivity_v1_selection.c